# TrustLap: equal-practice replication run (GPU)

For each of 3 training seeds: track banks → Basic AI (PBT 8 × 150M steps) → Teacher AI (PBT
8 × 150M, the same budget) → 5 Adaptive AI students (80M each) → the full evaluation on the same
test tracks; then one combined metrics file (seed 1 is the published one, fixed in advance).
About 4–5 hours on a T4. Every stage checkpoints and resumes, so if the session dies, run all again.

**Kaggle (recommended):** Settings → Accelerator → **GPU T4 ×2** (or P100), Internet **on**. Then
**Save Version → Save & Run All (Commit)**: it runs in the background, no tab needed. Outputs land in
`/kaggle/working/trustlap_rep.zip`.
**Colab:** Runtime → Change runtime type → T4 GPU → Run all (keep the tab open).


In [ ]:
import os

import jax

print("jax", jax.__version__, jax.devices())
assert jax.devices()[0].platform == "gpu", "No GPU visible: enable a GPU accelerator and rerun."
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
print("work dir:", WORK)

In [ ]:
%cd {WORK}
!rm -rf trustlap && git clone -q --depth 1 https://github.com/hitenjain22/trustlap.git
!pip install -q -e trustlap/sim
!cd trustlap && git log -1 --format='commit %h %s'

In [ ]:
# Persist runs outside the repo so a re-run resumes (Colab: optionally mount Drive first).
RUNS = os.path.join(WORK, "trustlap_runs")
os.makedirs(RUNS, exist_ok=True)
if not os.path.islink("trustlap/sim/runs"):
    !rm -rf trustlap/sim/runs && ln -s {RUNS} trustlap/sim/runs

In [ ]:
# The whole replication run (about 4–5 hours on a T4).
# Progress: trustlap_runs/rep/pipeline.log and each run's log.jsonl.
!cd trustlap/sim && bash scripts/replicate_pipeline.sh runs/rep 2>&1 | grep -v Warning | tail -n 60

In [ ]:
# Package the results for download (policies, logs, lineage, snapshots, metrics).
ZIP = f"{WORK}/trustlap_rep.zip"
!cd {RUNS} && zip -qr {ZIP} rep -x '*/checkpoint.msgpack' && ls -lh {ZIP}